# Coralscapes V1 on Moreton Bay BlueROV2 footage

Runs the pretrained **Coralscapes V1** SegFormer-B2 (`EPFL-ECEO/segformer-b2-finetuned-coralscapes-1024-1024`) on sampled frames of the Myora Bay onboard BlueROV2 video, **without any fine-tuning**, to observe how a model trained on Red Sea reef imagery behaves on turbid Moreton Bay footage.

Inference follows the approach recommended on the Coralscapes model card: the frame is resized so its short side is 1024 px, a 1024x1024 window slides across it, and logits from overlapping windows are averaged before the arg-max. The loader and inference code are in `utils/coral_utils.py`.

The published checkpoint's `config.json` contains malformed `id2label` and `label2id` entries (integers mapped to integers). `load_coralscapes_model` replaces them with a correctly typed 40-entry table (index 0 is `unlabeled`, indices 1 to 39 are the Coralscapes V1 classes), because deleting them would silently shrink the classifier head from 40 outputs to 2.

No ground truth is used here. The quantitative evaluation against expert annotations is in `zeroshot/`.

In [ ]:
import sys

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image

sys.path.append("../utils")   # shared modules live in the repository's utils folder
from coral_utils import load_coralscapes_model, segment_image, colourise_mask, coverage_stats

VIDEO_PATH = "../data/bluerov_videos/Myora-BlueROV.mp4"
FRAME_SAMPLE_EVERY_N_SEC = 50

# Apple GPU (MPS) when available, otherwise CPU
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

## Load the model

In [ ]:
preprocessor, model = load_coralscapes_model(device)
print("model outputs:", model.config.num_labels)

## Visualisation helper

In [ ]:
def show_segmentation(image, pred, alpha=0.6):
    colour_mask = colourise_mask(pred)
    image_np = np.array(image)
    overlay = (image_np * (1 - alpha) + colour_mask * alpha).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(image_np); axes[0].set_title("Original"); axes[0].axis("off")
    axes[1].imshow(colour_mask); axes[1].set_title("Segmentation"); axes[1].axis("off")
    axes[2].imshow(overlay); axes[2].set_title("Overlay"); axes[2].axis("off")
    plt.tight_layout()
    plt.show()

## Segment sampled frames

Extracts one frame every `FRAME_SAMPLE_EVERY_N_SEC` seconds, segments it and prints the five most common predicted classes. In the Moreton Bay footage `background` dominates (73 to 93% of pixels in most frames), which is the main observation motivating the ground-truth evaluation.

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise IOError(f"Could not open video: {VIDEO_PATH}")
fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval = int(fps * FRAME_SAMPLE_EVERY_N_SEC)

frame_idx = 0
first_frame_checked = False

while True:
    ret, frame_bgr = cap.read()
    if not ret:
        break

    if frame_idx % frame_interval == 0:
        image = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))

        pred = segment_image(image, preprocessor, model, device)
        show_segmentation(image, pred)

        # One-time sanity check on the class index range
        if not first_frame_checked:
            print(f"Predicted class id range: {pred.min()}-{pred.max()} (model has {model.config.num_labels} outputs)")
            first_frame_checked = True

        print(f"\n--- t={frame_idx/fps:.1f}s ---")
        for name, pct in coverage_stats(pred)[:5]:
            print(f"  {name:28s} {pct:5.2f}%")

    frame_idx += 1

cap.release()